# Calendar & Promotion Forecast Model Evaluation

## Business Objective

Evaluate whether advance-known calendar variables and
forecast-origin-safe historical promotion features improve
monthly retail sales forecasting accuracy.

## Forecasting Design

- Target: Monthly total sales
- Historical period: January 2013–July 2017
- Minimum training history: 36 months
- Forecast horizon: 3 months
- Validation: 17 expanding rolling-origin windows
- Evaluation cases: 51

## Existing Benchmarks

- 50/50 Blended: 8.36% WAPE
- ETS Additive: 8.48% WAPE

## Governance Rules

1. Fit each model using training information available at its forecast origin.
2. Do not use actual sales from validation months as predictors.
3. Do not use realized future promotion activity.
4. Do not train and evaluate a new model on the same 51 evaluation rows.
5. Compare models on identical forecast cases.
6. Report MAE, RMSE, WAPE, and forecast bias.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# Locate project root
PROJECT_ROOT = next(
    parent
    for parent in [Path.cwd(), *Path.cwd().parents]
    if (parent / "data" / "processed").exists()
)

# Required input files
required_files = [
    "data/processed/monthly_sales_complete.csv",
    "reports/retail_driver_analysis/monthly_sales_promotion_performance.csv",
    "reports/forecasting_feature_engineering/rolling_origin_calendar_promotion_features.csv",
]

for relative_path in required_files:
    file_path = PROJECT_ROOT / relative_path
    assert file_path.exists(), f"Missing file: {relative_path}"
    print("FOUND:", relative_path)

print("\nModel evaluation environment validation: PASSED")
print("Project root:", PROJECT_ROOT)
print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)

FOUND: data/processed/monthly_sales_complete.csv
FOUND: reports/retail_driver_analysis/monthly_sales_promotion_performance.csv
FOUND: reports/forecasting_feature_engineering/rolling_origin_calendar_promotion_features.csv

Model evaluation environment validation: PASSED
Project root: d:\GitHub\retail-sales-forecasting
Pandas version: 3.0.6
NumPy version: 2.5.3


## Step 2 — Load and Validate Modeling Inputs

Load historical monthly sales, recorded promotion activity,
and the existing rolling-origin evaluation cases.

Validate dates, monthly coverage, uniqueness, and sales
reconciliation before preparing model predictors.

In [2]:
# Load historical datasets
monthly_sales_df = pd.read_csv(
    PROJECT_ROOT / "data/processed/monthly_sales_complete.csv",
    parse_dates=["month"]
)

monthly_promotion_df = pd.read_csv(
    PROJECT_ROOT
    / "reports/retail_driver_analysis/monthly_sales_promotion_performance.csv",
    parse_dates=["month"]
)

evaluation_df = pd.read_csv(
    PROJECT_ROOT
    / "reports/forecasting_feature_engineering/rolling_origin_calendar_promotion_features.csv",
    parse_dates=["train_end", "forecast_month", "origin_month"]
)

# Validate expected dataset sizes
assert len(monthly_sales_df) == 55
assert len(monthly_promotion_df) == 55
assert len(evaluation_df) == 51

# Validate unique historical months
assert monthly_sales_df["month"].is_unique
assert monthly_promotion_df["month"].is_unique

# Validate date alignment
assert set(monthly_sales_df["month"]) == set(monthly_promotion_df["month"])

# Validate evaluation design
assert evaluation_df["window_id"].nunique() == 17
assert evaluation_df["forecast_horizon"].between(1, 3).all()

assert (
    evaluation_df.groupby("window_id").size() == 3
).all()

assert (
    evaluation_df["forecast_month"] > evaluation_df["train_end"]
).all()

# Reconcile monthly sales between historical files
sales_reconciliation_df = monthly_sales_df[
    ["month", "total_sales"]
].merge(
    monthly_promotion_df[["month", "monthly_sales"]],
    on="month",
    validate="one_to_one"
)

max_sales_difference = (
    sales_reconciliation_df["total_sales"]
    - sales_reconciliation_df["monthly_sales"]
).abs().max()

assert np.isclose(max_sales_difference, 0, atol=0.01)

print("Modeling input validation: PASSED")
print("Historical sales months:", len(monthly_sales_df))
print("Historical promotion months:", len(monthly_promotion_df))
print("Rolling forecast windows:", evaluation_df["window_id"].nunique())
print("Evaluation cases:", len(evaluation_df))
print("Maximum sales reconciliation difference:", max_sales_difference)

print("\nHistorical sales columns:")
print(monthly_sales_df.columns.tolist())

print("\nPromotion columns:")
print(monthly_promotion_df.columns.tolist())

display(evaluation_df.head(3))

Modeling input validation: PASSED
Historical sales months: 55
Historical promotion months: 55
Rolling forecast windows: 17
Evaluation cases: 51
Maximum sales reconciliation difference: 0.0

Historical sales columns:
['month', 'total_sales', 'source_rows', 'sales_dates_observed', 'stores_observed', 'families_observed', 'calendar_days', 'sales_closure_days', 'is_partial_month', 'sales_coverage_complete', 'is_forecast_training_period']

Promotion columns:
['month', 'monthly_sales', 'total_records', 'promotional_records', 'total_promoted_item_counts', 'promotion_record_share_pct', 'avg_sales_per_record']


,window_id,train_end,forecast_month,forecast_horizon,origin_month,origin_promotion_share_lag_1,origin_promotion_share_3m_avg,origin_promoted_items_3m_sum,origin_promotion_coverage_complete,calendar_year,calendar_month,calendar_quarter,days_in_month,is_december,is_february,total_sales,blended_forecast,ets_forecast
0,1,2015-12-01,2016-01-01,1,2015-12-01,22.343808,21.192511,376710,True,2016,1,1,31,0,0,2.397780e+07,2.107045e+07,2.346665e+07
1,1,2015-12-01,2016-02-01,2,2015-12-01,22.343808,21.192511,376710,True,2016,2,1,29,0,1,2.194741e+07,2.049319e+07,1.930117e+07
2,1,2015-12-01,2016-03-01,3,2015-12-01,22.343808,21.192511,376710,True,2016,3,1,31,0,0,2.313178e+07,2.142129e+07,2.421934e+07


## Step 3 — Construct the Monthly Modeling Dataset

Join historical monthly sales with recorded promotion activity.

Preserve one row per calendar month and validate sales totals,
promotion values, and chronological continuity.

Actual promotion activity from a forecast month must not
be used as a predictor for that month's forecast.

In [3]:
# Select only the required historical columns
sales_model_df = monthly_sales_df[
    ["month", "total_sales"]
].copy()

promotion_model_df = monthly_promotion_df[
    [
        "month",
        "promotional_records",
        "total_promoted_item_counts",
        "promotion_record_share_pct",
    ]
].copy()

# Join monthly sales and promotion information
modeling_monthly_df = sales_model_df.merge(
    promotion_model_df,
    on="month",
    how="inner",
    validate="one_to_one"
)

# Sort chronologically
modeling_monthly_df = (
    modeling_monthly_df
    .sort_values("month")
    .reset_index(drop=True)
)

# Flag historical months with uncertain promotion coverage
modeling_monthly_df["promotion_coverage_status"] = np.where(
    modeling_monthly_df["month"] < pd.Timestamp("2014-04-01"),
    "COVERAGE_UNCERTAIN",
    "RECORDED_PERIOD"
)

# Add advance-known calendar variables
modeling_monthly_df["calendar_month"] = (
    modeling_monthly_df["month"].dt.month
)

modeling_monthly_df["calendar_quarter"] = (
    modeling_monthly_df["month"].dt.quarter
)

modeling_monthly_df["days_in_month"] = (
    modeling_monthly_df["month"].dt.days_in_month
)

# Data quality validation
expected_months = pd.date_range(
    "2013-01-01",
    "2017-07-01",
    freq="MS"
)

assert len(modeling_monthly_df) == 55
assert modeling_monthly_df["month"].is_unique

assert modeling_monthly_df["month"].tolist() == expected_months.tolist()

assert modeling_monthly_df[
    [
        "total_sales",
        "promotional_records",
        "total_promoted_item_counts",
        "promotion_record_share_pct",
    ]
].notna().all().all()

assert modeling_monthly_df["promotion_record_share_pct"].between(
    0, 100
).all()

assert (
    modeling_monthly_df["promotion_coverage_status"]
    .eq("COVERAGE_UNCERTAIN")
    .sum()
    == 15
)

assert (
    modeling_monthly_df["promotion_coverage_status"]
    .eq("RECORDED_PERIOD")
    .sum()
    == 40
)

print("Monthly modeling dataset construction: PASSED")
print("Historical months:", len(modeling_monthly_df))
print("Columns:", len(modeling_monthly_df.columns))
print(
    "Promotion coverage uncertain:",
    modeling_monthly_df["promotion_coverage_status"]
    .eq("COVERAGE_UNCERTAIN").sum()
)
print(
    "Recorded promotion period:",
    modeling_monthly_df["promotion_coverage_status"]
    .eq("RECORDED_PERIOD").sum()
)

display(modeling_monthly_df.head(5))
display(modeling_monthly_df.tail(5))

Monthly modeling dataset construction: PASSED
Historical months: 55
Columns: 9
Promotion coverage uncertain: 15
Recorded promotion period: 40


,month,total_sales,promotional_records,total_promoted_item_counts,promotion_record_share_pct,promotion_coverage_status,calendar_month,calendar_quarter,days_in_month
0,2013-01-01,1.032762e+07,0,0,0.0,COVERAGE_UNCERTAIN,1,1,31
1,2013-02-01,9.658960e+06,0,0,0.0,COVERAGE_UNCERTAIN,2,1,28
2,2013-03-01,1.142850e+07,0,0,0.0,COVERAGE_UNCERTAIN,3,1,31
3,2013-04-01,1.099346e+07,0,0,0.0,COVERAGE_UNCERTAIN,4,2,30
4,2013-05-01,1.159770e+07,0,0,0.0,COVERAGE_UNCERTAIN,5,2,31


,month,total_sales,promotional_records,total_promoted_item_counts,promotion_record_share_pct,promotion_coverage_status,calendar_month,calendar_quarter,days_in_month
50,2017-03-01,2.670402e+07,23775,339291,43.037906,RECORDED_PERIOD,3,1,31
51,2017-04-01,2.589531e+07,24186,371042,45.241302,RECORDED_PERIOD,4,2,30
52,2017-05-01,2.691185e+07,27172,414646,49.187213,RECORDED_PERIOD,5,2,31
53,2017-06-01,2.568282e+07,23724,394582,44.377104,RECORDED_PERIOD,6,2,30
54,2017-07-01,2.701148e+07,23373,409128,42.310199,RECORDED_PERIOD,7,3,31


## Step 4 — Construct Forecast-Safe Training Examples

Create supervised learning examples using historical
forecast origins and horizons of one, two, and three months.

Use only historical information available at each origin
and advance-known calendar features for target months.

Actual target-month sales are training labels, not predictors.

In [4]:
def build_training_examples(history_df):
    """
    Construct forecast-safe training examples.

    Each example uses:
    - Historical sales at its forecast origin
    - Historical promotion activity through that origin
    - Calendar information for its target month

    Only target months within history_df are included.
    """

    history = history_df.sort_values("month").reset_index(drop=True)

    records = []

    # Require at least 12 months of historical sales
    for origin_idx in range(11, len(history) - 1):

        origin = history.iloc[origin_idx]

        # Historical promotion information through origin
        promotion_history = history.iloc[:origin_idx + 1]
        latest_three = promotion_history.tail(3)

        # Exclude examples with uncertain recent promotion coverage
        if not latest_three[
            "promotion_coverage_status"
        ].eq("RECORDED_PERIOD").all():
            continue

        for horizon in [1, 2, 3]:

            target_idx = origin_idx + horizon

            if target_idx >= len(history):
                continue

            target = history.iloc[target_idx]

            records.append({
                "origin_month": origin["month"],
                "target_month": target["month"],
                "forecast_horizon": horizon,

                "origin_sales_lag_1": origin["total_sales"],

                "origin_sales_3m_avg":
                    promotion_history["total_sales"].tail(3).mean(),

                "origin_promotion_share_lag_1":
                    origin["promotion_record_share_pct"],

                "origin_promotion_share_3m_avg":
                    latest_three["promotion_record_share_pct"].mean(),

                "origin_promoted_items_3m_sum":
                    latest_three["total_promoted_item_counts"].sum(),

                "calendar_month": target["calendar_month"],
                "calendar_quarter": target["calendar_quarter"],
                "days_in_month": target["days_in_month"],

                "is_december": int(target["calendar_month"] == 12),
                "is_february": int(target["calendar_month"] == 2),

                # Label only — never use as predictor
                "target_sales": target["total_sales"],
            })

    return pd.DataFrame(records)


# Construct examples using all available history
training_examples_df = build_training_examples(modeling_monthly_df)

# Validate training examples
assert not training_examples_df.empty

assert (
    training_examples_df["target_month"]
    > training_examples_df["origin_month"]
).all()

assert training_examples_df["forecast_horizon"].between(1, 3).all()

assert training_examples_df[
    ["origin_month", "target_month"]
].duplicated().sum() == 0

print("Forecast-safe training example construction: PASSED")
print("Training examples:", len(training_examples_df))
print("Columns:", len(training_examples_df.columns))
print(
    "First training origin:",
    training_examples_df["origin_month"].min().date()
)
print(
    "Last training target:",
    training_examples_df["target_month"].max().date()
)

display(training_examples_df.head(6))

Forecast-safe training example construction: PASSED
Training examples: 108
Columns: 14
First training origin: 2014-06-01
Last training target: 2017-07-01


,origin_month,target_month,forecast_horizon,origin_sales_lag_1,origin_sales_3m_avg,origin_promotion_share_lag_1,origin_promotion_share_3m_avg,origin_promoted_items_3m_sum,calendar_month,calendar_quarter,days_in_month,is_december,is_february,target_sales
0,2014-06-01,2014-07-01,1,1.331996e+07,1.318700e+07,8.168724,4.156781,35010,7,3,31,0,0,1.942189e+07
1,2014-06-01,2014-08-01,2,1.331996e+07,1.318700e+07,8.168724,4.156781,35010,8,3,31,0,0,1.388518e+07
2,2014-06-01,2014-09-01,3,1.331996e+07,1.318700e+07,8.168724,4.156781,35010,9,3,30,0,0,2.002242e+07
3,2014-07-01,2014-08-01,1,1.942189e+07,1.537388e+07,16.201441,9.308477,102736,8,3,31,0,0,1.388518e+07
4,2014-07-01,2014-09-01,2,1.942189e+07,1.537388e+07,16.201441,9.308477,102736,9,3,30,0,0,2.002242e+07
5,2014-07-01,2014-10-01,3,1.942189e+07,1.537388e+07,16.201441,9.308477,102736,10,4,31,0,0,2.039610e+07


## Step 5 — Validate Rolling-Origin Training Cutoffs

Build a separate historical training dataset for each
of the 17 forecast windows.

Verify that all training origins and target sales dates
occur no later than the window's training cutoff.

Do not fit forecasting models in this step.

In [5]:
# Create a training-data audit for each rolling forecast window
training_window_records = []

for window_id, group in evaluation_df.groupby("window_id"):

    train_end = group["train_end"].iloc[0]

    # Only historical months available at this forecast origin
    historical_data = modeling_monthly_df.loc[
        modeling_monthly_df["month"] <= train_end
    ].copy()

    # Construct examples using the historical cutoff
    window_training_df = build_training_examples(historical_data)

    assert not window_training_df.empty

    # No training origin or target may exceed the cutoff
    assert (window_training_df["origin_month"] <= train_end).all()
    assert (window_training_df["target_month"] <= train_end).all()

    # Validate the three forecast horizons
    assert window_training_df["forecast_horizon"].between(1, 3).all()

    # Ensure validation months occur after training
    assert (group["forecast_month"] > train_end).all()

    training_window_records.append({
        "window_id": window_id,
        "train_end": train_end,
        "historical_months": len(historical_data),
        "training_examples": len(window_training_df),
        "first_training_origin":
            window_training_df["origin_month"].min(),
        "last_training_target":
            window_training_df["target_month"].max(),
        "validation_start": group["forecast_month"].min(),
        "validation_end": group["forecast_month"].max(),
    })

training_window_audit_df = pd.DataFrame(training_window_records)

# Final validations
assert len(training_window_audit_df) == 17

assert (
    training_window_audit_df["last_training_target"]
    <= training_window_audit_df["train_end"]
).all()

assert (
    training_window_audit_df["validation_start"]
    > training_window_audit_df["train_end"]
).all()

assert training_window_audit_df["training_examples"].gt(0).all()

print("Rolling-origin training cutoff validation: PASSED")
print("Forecast windows:", len(training_window_audit_df))
print(
    "Training examples in first window:",
    training_window_audit_df["training_examples"].iloc[0]
)
print(
    "Training examples in last window:",
    training_window_audit_df["training_examples"].iloc[-1]
)
print("Training targets beyond cutoff: 0")

display(training_window_audit_df.head(5))
display(training_window_audit_df.tail(5))

Rolling-origin training cutoff validation: PASSED
Forecast windows: 17
Training examples in first window: 51
Training examples in last window: 99
Training targets beyond cutoff: 0


,window_id,train_end,historical_months,training_examples,first_training_origin,last_training_target,validation_start,validation_end
0,1,2015-12-01,36,51,2014-06-01,2015-12-01,2016-01-01,2016-03-01
1,2,2016-01-01,37,54,2014-06-01,2016-01-01,2016-02-01,2016-04-01
2,3,2016-02-01,38,57,2014-06-01,2016-02-01,2016-03-01,2016-05-01
3,4,2016-03-01,39,60,2014-06-01,2016-03-01,2016-04-01,2016-06-01
4,5,2016-04-01,40,63,2014-06-01,2016-04-01,2016-05-01,2016-07-01


,window_id,train_end,historical_months,training_examples,first_training_origin,last_training_target,validation_start,validation_end
12,13,2016-12-01,48,87,2014-06-01,2016-12-01,2017-01-01,2017-03-01
13,14,2017-01-01,49,90,2014-06-01,2017-01-01,2017-02-01,2017-04-01
14,15,2017-02-01,50,93,2014-06-01,2017-02-01,2017-03-01,2017-05-01
15,16,2017-03-01,51,96,2014-06-01,2017-03-01,2017-04-01,2017-06-01
16,17,2017-04-01,52,99,2014-06-01,2017-04-01,2017-05-01,2017-07-01


## Step 6 — Define Forecast Model Predictors

Define a calendar-only model and a calendar-plus-promotion model.

Use historical sales available at each forecast origin,
advance-known calendar features, and historical promotion
features where applicable.

Exclude target sales and all realized future information.

In [6]:
# Import modeling tools
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Historical sales and advance-known calendar predictors
calendar_feature_columns = [
    "forecast_horizon",
    "origin_sales_lag_1",
    "origin_sales_3m_avg",
    "calendar_month",
    "calendar_quarter",
    "days_in_month",
    "is_december",
    "is_february",
]

# Promotion information available at the forecast origin
promotion_feature_columns = [
    "origin_promotion_share_lag_1",
    "origin_promotion_share_3m_avg",
    "origin_promoted_items_3m_sum",
]

# Model feature sets
model_feature_sets = {
    "Calendar Only": calendar_feature_columns,
    "Calendar + Promotion": (
        calendar_feature_columns + promotion_feature_columns
    ),
}

# Validate predictor definitions
assert len(calendar_feature_columns) == 8
assert len(promotion_feature_columns) == 3

for model_name, features in model_feature_sets.items():

    assert len(features) == len(set(features))

    assert set(features).issubset(training_examples_df.columns)

    assert "target_sales" not in features
    assert "target_month" not in features
    assert "origin_month" not in features

    print(f"{model_name}: {len(features)} predictors")

# Create an unfitted model pipeline
def create_ridge_pipeline():
    return Pipeline([
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=10.0)),
    ])

print("\nForecast model predictor validation: PASSED")

Calendar Only: 8 predictors
Calendar + Promotion: 11 predictors

Forecast model predictor validation: PASSED


## Step 7 — Rolling-Origin Ridge Regression Forecasts

Train Calendar Only and Calendar + Promotion Ridge Regression
models independently within each of the 17 rolling windows.

Fit both the StandardScaler and Ridge model using only
historical training examples available at each forecast origin.

Generate three forecasts per window and validate the
51 matched forecast cases.

Do not calculate model accuracy until forecast construction
and leakage checks have passed.

In [7]:
# Store forecasts from both models
ridge_forecast_records = []

# Process the 17 historical forecast windows
for window_id, group in evaluation_df.groupby("window_id"):

    train_end = group["train_end"].iloc[0]

    # Select only data available at this forecast origin
    historical_data = modeling_monthly_df.loc[
        modeling_monthly_df["month"] <= train_end
    ].copy()

    # Build historical training examples
    window_training_df = build_training_examples(
        historical_data
    )

    # Prepare the three future forecast cases
    window_validation_df = group.sort_values(
        "forecast_horizon"
    ).copy()

    assert len(window_validation_df) == 3

    # Historical sales features at the forecast origin
    latest_sales = historical_data["total_sales"].iloc[-1]

    recent_sales_avg = (
        historical_data["total_sales"].tail(3).mean()
    )

    window_validation_df["origin_sales_lag_1"] = latest_sales
    window_validation_df["origin_sales_3m_avg"] = recent_sales_avg

    # Train and predict separately for each model
    for model_name, feature_columns in model_feature_sets.items():

        model = create_ridge_pipeline()

        X_train = window_training_df[feature_columns]
        y_train = window_training_df["target_sales"]

        X_validation = window_validation_df[feature_columns]

        # Fit only on historical training examples
        model.fit(X_train, y_train)

        # Predict three future months
        predictions = model.predict(X_validation)

        for row, prediction in zip(
            window_validation_df.itertuples(index=False),
            predictions
        ):
            ridge_forecast_records.append({
                "window_id": window_id,
                "train_end": train_end,
                "forecast_month": row.forecast_month,
                "forecast_horizon": row.forecast_horizon,
                "model": model_name,
                "actual_sales": row.total_sales,
                "predicted_sales": float(prediction),
                "training_examples": len(window_training_df),
            })

# Convert results to DataFrame
ridge_forecasts_df = pd.DataFrame(ridge_forecast_records)

# Validate forecast structure
assert len(ridge_forecasts_df) == 102

assert ridge_forecasts_df["window_id"].nunique() == 17

assert (
    ridge_forecasts_df.groupby("model").size() == 51
).all()

assert not ridge_forecasts_df[
    ["actual_sales", "predicted_sales"]
].isna().any().any()

assert np.isfinite(
    ridge_forecasts_df["predicted_sales"]
).all()

assert (
    ridge_forecasts_df["forecast_month"]
    > ridge_forecasts_df["train_end"]
).all()

assert not ridge_forecasts_df[
    ["window_id", "forecast_month", "model"]
].duplicated().any()

print("Rolling-origin Ridge forecasting: PASSED")
print("Forecast windows:", ridge_forecasts_df["window_id"].nunique())
print("Models:", ridge_forecasts_df["model"].nunique())
print("Forecasts per model:", 51)
print("Total forecast records:", len(ridge_forecasts_df))

display(ridge_forecasts_df.head(9))

Rolling-origin Ridge forecasting: PASSED
Forecast windows: 17
Models: 2
Forecasts per model: 51
Total forecast records: 102


,window_id,train_end,forecast_month,forecast_horizon,model,actual_sales,predicted_sales,training_examples
0,1,2015-12-01,2016-01-01,1,Calendar Only,2.397780e+07,1.747551e+07,51
1,1,2015-12-01,2016-02-01,2,Calendar Only,2.194741e+07,1.563638e+07,51
2,1,2015-12-01,2016-03-01,3,Calendar Only,2.313178e+07,1.875459e+07,51
3,1,2015-12-01,2016-01-01,1,Calendar + Promotion,2.397780e+07,1.615498e+07,51
4,1,2015-12-01,2016-02-01,2,Calendar + Promotion,2.194741e+07,1.479011e+07,51
5,1,2015-12-01,2016-03-01,3,Calendar + Promotion,2.313178e+07,1.743424e+07,51
6,2,2016-01-01,2016-02-01,1,Calendar Only,2.194741e+07,1.592873e+07,54
7,2,2016-01-01,2016-03-01,2,Calendar Only,2.313178e+07,2.037461e+07,54
8,2,2016-01-01,2016-04-01,3,Calendar Only,2.596303e+07,2.129506e+07,54


## Step 8 — Ridge Forecast Quality and Alignment Validation

Validate the 102 rolling-origin Ridge forecasts.

Check for missing, nonfinite, and negative predictions.

Reconcile forecast actuals against the original evaluation
dataset and verify that both models predict identical cases.

In [8]:
# 1. Check prediction quality
missing_predictions = ridge_forecasts_df["predicted_sales"].isna().sum()

negative_predictions = (
    ridge_forecasts_df["predicted_sales"] < 0
).sum()

zero_predictions = (
    ridge_forecasts_df["predicted_sales"] == 0
).sum()

nonfinite_predictions = (
    ~np.isfinite(ridge_forecasts_df["predicted_sales"])
).sum()

# 2. Reconcile actual sales against original evaluation data
forecast_alignment_df = ridge_forecasts_df.merge(
    evaluation_df[
        ["window_id", "train_end", "forecast_month", "total_sales"]
    ],
    on=["window_id", "train_end", "forecast_month"],
    how="left",
    validate="many_to_one",
    indicator=True
)

assert forecast_alignment_df["_merge"].eq("both").all()

max_actual_difference = (
    forecast_alignment_df["actual_sales"]
    - forecast_alignment_df["total_sales"]
).abs().max()

# 3. Check identical forecast cases for both models
calendar_cases = set(
    map(
        tuple,
        ridge_forecasts_df.loc[
            ridge_forecasts_df["model"] == "Calendar Only",
            ["window_id", "forecast_month"]
        ].to_numpy()
    )
)

promotion_cases = set(
    map(
        tuple,
        ridge_forecasts_df.loc[
            ridge_forecasts_df["model"] == "Calendar + Promotion",
            ["window_id", "forecast_month"]
        ].to_numpy()
    )
)

# 4. Validate
assert missing_predictions == 0
assert nonfinite_predictions == 0
assert np.isclose(max_actual_difference, 0, atol=0.01)
assert calendar_cases == promotion_cases
assert len(calendar_cases) == 51

print("Ridge forecast quality and alignment validation: PASSED")
print("Missing predictions:", missing_predictions)
print("Nonfinite predictions:", nonfinite_predictions)
print("Negative predictions:", negative_predictions)
print("Zero predictions:", zero_predictions)
print("Maximum actual-sales difference:", max_actual_difference)
print("Matched cases per model:", len(calendar_cases))

Ridge forecast quality and alignment validation: PASSED
Missing predictions: 0
Nonfinite predictions: 0
Negative predictions: 0
Zero predictions: 0
Maximum actual-sales difference: 0.0
Matched cases per model: 51


## Step 9 — Ridge Forecast Accuracy Evaluation

Evaluate both Ridge Regression models using the same
51 rolling-origin forecast cases.

Calculate:
- Mean Absolute Error (MAE)
- Root Mean Squared Error (RMSE)
- Weighted Absolute Percentage Error (WAPE)
- Forecast Bias Percentage

A positive bias indicates overforecasting.
A negative bias indicates underforecasting.

These are overlapping rolling-origin forecast cases,
not 51 independent calendar months.

In [9]:
# Calculate forecast errors
ridge_forecasts_df["forecast_error"] = (
    ridge_forecasts_df["predicted_sales"]
    - ridge_forecasts_df["actual_sales"]
)

ridge_forecasts_df["absolute_error"] = (
    ridge_forecasts_df["forecast_error"].abs()
)

ridge_forecasts_df["squared_error"] = (
    ridge_forecasts_df["forecast_error"] ** 2
)

# Calculate accuracy metrics for each model
ridge_accuracy_records = []

for model_name, group in ridge_forecasts_df.groupby("model"):

    actual_total = group["actual_sales"].sum()

    mae = group["absolute_error"].mean()

    rmse = np.sqrt(
        group["squared_error"].mean()
    )

    wape = (
        group["absolute_error"].sum()
        / actual_total
        * 100
    )

    bias = (
        group["forecast_error"].sum()
        / actual_total
        * 100
    )

    ridge_accuracy_records.append({
        "model": model_name,
        "forecast_cases": len(group),
        "MAE": mae,
        "RMSE": rmse,
        "WAPE_pct": wape,
        "Bias_pct": bias,
    })

ridge_accuracy_df = pd.DataFrame(
    ridge_accuracy_records
).sort_values("WAPE_pct").reset_index(drop=True)

# Validate evaluation results
assert len(ridge_accuracy_df) == 2

assert (
    ridge_accuracy_df["forecast_cases"] == 51
).all()

assert ridge_accuracy_df[
    ["MAE", "RMSE", "WAPE_pct", "Bias_pct"]
].notna().all().all()

assert np.isfinite(
    ridge_accuracy_df[
        ["MAE", "RMSE", "WAPE_pct", "Bias_pct"]
    ].to_numpy()
).all()

assert (ridge_accuracy_df["MAE"] >= 0).all()
assert (ridge_accuracy_df["RMSE"] >= 0).all()
assert (ridge_accuracy_df["WAPE_pct"] >= 0).all()

print("Ridge forecast accuracy evaluation: PASSED")
print("Models evaluated:", len(ridge_accuracy_df))
print("Forecast cases per model: 51")

display(
    ridge_accuracy_df.round({
        "MAE": 2,
        "RMSE": 2,
        "WAPE_pct": 2,
        "Bias_pct": 2,
    })
)

Ridge forecast accuracy evaluation: PASSED
Models evaluated: 2
Forecast cases per model: 51


,model,forecast_cases,MAE,RMSE,WAPE_pct,Bias_pct
0,Calendar Only,51,2475681.41,3007249.48,10.01,-7.85
1,Calendar + Promotion,51,3028155.12,3789357.27,12.24,-1.34


## Step 10 — Matched-Case Forecast Benchmark Comparison

Compare both Ridge Regression models against the existing
50/50 Blended and ETS Additive forecasting benchmarks.

Use the same 51 rolling-origin forecast cases.

Independently calculate MAE, RMSE, WAPE, and bias
from the saved benchmark predictions.

Avoid drawing conclusions from rounded historical metrics.

In [10]:
# Existing benchmark forecast columns
benchmark_models = {
    "50/50 Blended": "blended_forecast",
    "ETS Additive": "ets_forecast",
}

benchmark_accuracy_records = []

for model_name, forecast_column in benchmark_models.items():

    actual = evaluation_df["total_sales"].to_numpy()
    predicted = evaluation_df[forecast_column].to_numpy()

    # Forecast minus actual
    errors = predicted - actual

    mae = np.abs(errors).mean()

    rmse = np.sqrt(
        np.mean(errors ** 2)
    )

    wape = (
        np.abs(errors).sum()
        / actual.sum()
        * 100
    )

    bias = (
        errors.sum()
        / actual.sum()
        * 100
    )

    benchmark_accuracy_records.append({
        "model": model_name,
        "forecast_cases": len(actual),
        "MAE": mae,
        "RMSE": rmse,
        "WAPE_pct": wape,
        "Bias_pct": bias,
    })

benchmark_accuracy_df = pd.DataFrame(
    benchmark_accuracy_records
)

# Combine benchmarks with the two Ridge models
four_model_comparison_df = pd.concat(
    [
        benchmark_accuracy_df,
        ridge_accuracy_df,
    ],
    ignore_index=True
).sort_values(
    "WAPE_pct"
).reset_index(drop=True)

# Validation
assert len(four_model_comparison_df) == 4

assert (
    four_model_comparison_df["forecast_cases"] == 51
).all()

assert four_model_comparison_df["model"].is_unique

assert np.isfinite(
    four_model_comparison_df[
        ["MAE", "RMSE", "WAPE_pct", "Bias_pct"]
    ].to_numpy()
).all()

# Historical benchmark reconciliation
assert np.isclose(
    benchmark_accuracy_df.loc[
        benchmark_accuracy_df["model"] == "50/50 Blended",
        "WAPE_pct"
    ].iloc[0],
    8.36,
    atol=0.01
)

assert np.isclose(
    benchmark_accuracy_df.loc[
        benchmark_accuracy_df["model"] == "ETS Additive",
        "WAPE_pct"
    ].iloc[0],
    8.48,
    atol=0.01
)

print("Four-model matched-case comparison: PASSED")
print("Models compared:", len(four_model_comparison_df))
print("Forecast cases per model: 51")

display(
    four_model_comparison_df.round({
        "MAE": 2,
        "RMSE": 2,
        "WAPE_pct": 2,
        "Bias_pct": 2,
    })
)

Four-model matched-case comparison: PASSED
Models compared: 4
Forecast cases per model: 51


,model,forecast_cases,MAE,RMSE,WAPE_pct,Bias_pct
0,50/50 Blended,51,2067404.61,2737131.72,8.36,-6.45
1,ETS Additive,51,2098242.04,2542024.93,8.48,0.96
2,Calendar Only,51,2475681.41,3007249.48,10.01,-7.85
3,Calendar + Promotion,51,3028155.12,3789357.27,12.24,-1.34


## Step 11 — Promotion Feature Error Investigation

Compare Calendar Only and Calendar + Promotion Ridge forecasts
on identical rolling-origin forecast cases.

Identify how often promotion features improve or worsen
absolute forecast errors.

Investigate the largest deteriorations without assuming
that promotion activity caused the errors.

In [11]:
# Compare Ridge models on identical forecast cases
ridge_error_comparison_df = ridge_forecasts_df.pivot(
    index=[
        "window_id",
        "train_end",
        "forecast_month",
        "forecast_horizon",
        "actual_sales",
    ],
    columns="model",
    values="absolute_error",
).reset_index()

ridge_error_comparison_df.columns.name = None

# Positive values mean promotion features made errors worse
ridge_error_comparison_df["promotion_error_increase"] = (
    ridge_error_comparison_df["Calendar + Promotion"]
    - ridge_error_comparison_df["Calendar Only"]
)

# Count improvements and deteriorations
improved_cases = (
    ridge_error_comparison_df["promotion_error_increase"] < 0
).sum()

worsened_cases = (
    ridge_error_comparison_df["promotion_error_increase"] > 0
).sum()

unchanged_cases = (
    ridge_error_comparison_df["promotion_error_increase"] == 0
).sum()

# Validate matched-case comparison
assert len(ridge_error_comparison_df) == 51

assert (
    improved_cases + worsened_cases + unchanged_cases
    == 51
)

assert np.isclose(
    ridge_error_comparison_df["Calendar Only"].mean(),
    ridge_accuracy_df.loc[
        ridge_accuracy_df["model"] == "Calendar Only",
        "MAE"
    ].iloc[0]
)

assert np.isclose(
    ridge_error_comparison_df["Calendar + Promotion"].mean(),
    ridge_accuracy_df.loc[
        ridge_accuracy_df["model"] == "Calendar + Promotion",
        "MAE"
    ].iloc[0]
)

print("Promotion feature error investigation: PASSED")
print("Matched forecast cases:", len(ridge_error_comparison_df))
print("Cases improved by promotion features:", improved_cases)
print("Cases worsened by promotion features:", worsened_cases)
print("Cases unchanged:", unchanged_cases)

print("\nFive largest error increases:")
display(
    ridge_error_comparison_df.sort_values(
        "promotion_error_increase",
        ascending=False
    ).head(5)
)

Promotion feature error investigation: PASSED
Matched forecast cases: 51
Cases improved by promotion features: 30
Cases worsened by promotion features: 21
Cases unchanged: 0

Five largest error increases:


,window_id,train_end,forecast_month,forecast_horizon,actual_sales,Calendar + Promotion,Calendar Only,promotion_error_increase
17,6,2016-05-01,2016-08-01,3,2.245241e+07,8.875803e+06,2.562270e+06,6.313534e+06
16,6,2016-05-01,2016-07-01,2,2.346267e+07,7.217809e+06,1.023703e+06,6.194106e+06
15,6,2016-05-01,2016-06-01,1,2.220922e+07,6.964600e+06,9.061860e+05,6.058414e+06
19,7,2016-06-01,2016-08-01,2,2.245241e+07,5.949091e+06,1.520993e+06,4.428098e+06
20,7,2016-06-01,2016-09-01,3,2.241745e+07,6.073879e+06,1.707667e+06,4.366212e+06


## Step 12 — Forecast Error Analysis by Horizon

Evaluate Calendar Only and Calendar + Promotion Ridge models
separately for forecast horizons 1, 2, and 3.

Calculate MAE, RMSE, WAPE, and bias for each horizon.

Identify whether promotion features improve short-term
forecasts but worsen longer-horizon predictions.

In [12]:
# Evaluate accuracy separately for each forecast horizon
horizon_accuracy_records = []

for (model_name, horizon), group in ridge_forecasts_df.groupby(
    ["model", "forecast_horizon"]
):

    actual_total = group["actual_sales"].sum()

    horizon_accuracy_records.append({
        "model": model_name,
        "forecast_horizon": horizon,
        "forecast_cases": len(group),

        "MAE": group["absolute_error"].mean(),

        "RMSE": np.sqrt(
            group["squared_error"].mean()
        ),

        "WAPE_pct": (
            group["absolute_error"].sum()
            / actual_total * 100
        ),

        "Bias_pct": (
            group["forecast_error"].sum()
            / actual_total * 100
        ),
    })

horizon_accuracy_df = pd.DataFrame(
    horizon_accuracy_records
).sort_values(
    ["forecast_horizon", "model"]
).reset_index(drop=True)

# Validate results
assert len(horizon_accuracy_df) == 6

assert (
    horizon_accuracy_df["forecast_cases"] == 17
).all()

assert set(horizon_accuracy_df["forecast_horizon"]) == {1, 2, 3}

assert horizon_accuracy_df[
    ["MAE", "RMSE", "WAPE_pct", "Bias_pct"]
].notna().all().all()

# Reconcile horizon-level absolute errors to overall MAE
for model_name, group in ridge_forecasts_df.groupby("model"):

    horizon_mae = horizon_accuracy_df.loc[
        horizon_accuracy_df["model"] == model_name,
        "MAE"
    ].mean()

    overall_mae = group["absolute_error"].mean()

    assert np.isclose(horizon_mae, overall_mae)

print("Forecast horizon error analysis: PASSED")
print("Models evaluated:", horizon_accuracy_df["model"].nunique())
print("Forecast horizons:", sorted(horizon_accuracy_df["forecast_horizon"].unique()))
print("Forecast cases per model per horizon: 17")

display(
    horizon_accuracy_df.round({
        "MAE": 2,
        "RMSE": 2,
        "WAPE_pct": 2,
        "Bias_pct": 2,
    })
)

Forecast horizon error analysis: PASSED
Models evaluated: 2
Forecast horizons: [np.int64(1), np.int64(2), np.int64(3)]
Forecast cases per model per horizon: 17


,model,forecast_horizon,forecast_cases,MAE,RMSE,WAPE_pct,Bias_pct
0,Calendar + Promotion,1,17,2875126.75,3586726.16,11.70,-2.71
1,Calendar Only,1,17,2568679.21,3223319.42,10.45,-9.27
2,Calendar + Promotion,2,17,3005828.12,3773014.83,12.18,-1.14
3,Calendar Only,2,17,2457908.47,2965079.33,9.96,-7.66
4,Calendar + Promotion,3,17,3203510.48,3997179.02,12.83,-0.20
5,Calendar Only,3,17,2400456.54,2819426.32,9.61,-6.65


## Step 13 — Forecast Error Contribution by Window

Compare absolute forecast errors for both Ridge models
within each of the 17 rolling-origin windows.

Identify the windows contributing most to the promotion
model's higher overall forecasting error.

Reconcile window-level errors with the overall model
accuracy results.

Do not interpret error concentration as evidence of causation.

In [13]:
# Calculate total absolute error by model and window
window_error_df = (
    ridge_forecasts_df
    .groupby(
        ["window_id", "train_end", "model"],
        as_index=False
    )
    .agg(
        forecast_cases=("forecast_month", "size"),
        total_absolute_error=("absolute_error", "sum"),
    )
)

# Compare models side by side
window_error_comparison_df = (
    window_error_df
    .pivot(
        index=["window_id", "train_end"],
        columns="model",
        values="total_absolute_error"
    )
    .reset_index()
)

window_error_comparison_df.columns.name = None

# Positive difference means promotion features increased error
window_error_comparison_df["additional_absolute_error"] = (
    window_error_comparison_df["Calendar + Promotion"]
    - window_error_comparison_df["Calendar Only"]
)

# Reconcile with the overall forecast evaluation
overall_additional_error = (
    ridge_forecasts_df.loc[
        ridge_forecasts_df["model"] == "Calendar + Promotion",
        "absolute_error"
    ].sum()
    -
    ridge_forecasts_df.loc[
        ridge_forecasts_df["model"] == "Calendar Only",
        "absolute_error"
    ].sum()
)

assert len(window_error_comparison_df) == 17

assert np.isclose(
    window_error_comparison_df["additional_absolute_error"].sum(),
    overall_additional_error
)

assert (
    window_error_df["forecast_cases"] == 3
).all()

worsened_windows = (
    window_error_comparison_df["additional_absolute_error"] > 0
).sum()

improved_windows = (
    window_error_comparison_df["additional_absolute_error"] < 0
).sum()

unchanged_windows = (
    window_error_comparison_df["additional_absolute_error"] == 0
).sum()

assert (
    worsened_windows + improved_windows + unchanged_windows
    == 17
)

print("Forecast window error contribution analysis: PASSED")
print("Windows analyzed:", len(window_error_comparison_df))
print("Windows worsened:", worsened_windows)
print("Windows improved:", improved_windows)
print("Windows unchanged:", unchanged_windows)
print(
    "Overall additional absolute error:",
    round(overall_additional_error, 2)
)

print("\nFive largest error-contributing windows:")

display(
    window_error_comparison_df
    .sort_values(
        "additional_absolute_error",
        ascending=False
    )
    .head(5)
    .round(2)
)

Forecast window error contribution analysis: PASSED
Windows analyzed: 17
Windows worsened: 7
Windows improved: 10
Windows unchanged: 0
Overall additional absolute error: 28176159.26

Five largest error-contributing windows:


C:\Users\Dell\AppData\Local\Temp\ipykernel_1524\2080727772.py:93: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  .round(2)


,window_id,train_end,Calendar + Promotion,Calendar Only,additional_absolute_error
5,6,2016-05-01,23058212.64,4492158.93,18566053.71
6,7,2016-06-01,16352313.25,3245858.01,13106455.25
4,5,2016-04-01,11936706.35,3797101.15,8139605.20
7,8,2016-07-01,11632967.49,3510974.55,8121992.94
0,1,2015-12-01,20677667.62,17190510.89,3487156.73


In [14]:
# Select the five windows with the largest additional errors
top_error_windows_df = (
    window_error_comparison_df
    .sort_values(
        "additional_absolute_error",
        ascending=False
    )
    .head(5)
    .copy()
)

# Round only numeric error columns
error_columns = [
    "Calendar + Promotion",
    "Calendar Only",
    "additional_absolute_error",
]

top_error_windows_df[error_columns] = (
    top_error_windows_df[error_columns].round(2)
)

# Calculate how much positive deterioration
# comes from these five windows
total_positive_deterioration = (
    window_error_comparison_df.loc[
        window_error_comparison_df["additional_absolute_error"] > 0,
        "additional_absolute_error"
    ].sum()
)

top_five_positive_deterioration = (
    top_error_windows_df[
        "additional_absolute_error"
    ].clip(lower=0).sum()
)

top_five_share_pct = (
    top_five_positive_deterioration
    / total_positive_deterioration
    * 100
)

# Validate
assert len(top_error_windows_df) == 5
assert total_positive_deterioration > 0
assert 0 <= top_five_share_pct <= 100

print("Top error window investigation: PASSED")

print(
    "Total positive deterioration:",
    round(total_positive_deterioration, 2)
)

print(
    "Top five share of positive deterioration:",
    round(top_five_share_pct, 2),
    "%"
)

display(top_error_windows_df)

Top error window investigation: PASSED
Total positive deterioration: 54261670.07
Top five share of positive deterioration: 94.77 %


,window_id,train_end,Calendar + Promotion,Calendar Only,additional_absolute_error
5,6,2016-05-01,23058212.64,4492158.93,18566053.71
6,7,2016-06-01,16352313.25,3245858.01,13106455.25
4,5,2016-04-01,11936706.35,3797101.15,8139605.20
7,8,2016-07-01,11632967.49,3510974.55,8121992.94
0,1,2015-12-01,20677667.62,17190510.89,3487156.73


## Step 14 — Historical Promotion Signal Investigation

Investigate historical promotion signals at forecast origins
associated with the largest Ridge model error increases.

Compare origin-level promotion share and promoted-item counts
with the preceding three-month averages.

Use only information available at each forecast origin.

Do not infer that changes in promotion activity caused
the forecast errors.

In [15]:
# One row per forecast origin
origin_signal_df = (
    evaluation_df[
        [
            "window_id",
            "train_end",
            "origin_promotion_share_lag_1",
            "origin_promotion_share_3m_avg",
            "origin_promoted_items_3m_sum",
        ]
    ]
    .drop_duplicates(subset=["window_id"])
    .copy()
)

# Add the window-level error contribution
origin_signal_audit_df = origin_signal_df.merge(
    window_error_comparison_df[
        [
            "window_id",
            "additional_absolute_error",
        ]
    ],
    on="window_id",
    how="inner",
    validate="one_to_one"
)

# Compare latest promotion share with its trailing average
origin_signal_audit_df["promotion_share_vs_3m_avg_pp"] = (
    origin_signal_audit_df["origin_promotion_share_lag_1"]
    - origin_signal_audit_df["origin_promotion_share_3m_avg"]
)

# Identify the four consecutive high-error windows
focus_windows_df = (
    origin_signal_audit_df.loc[
        origin_signal_audit_df["window_id"].isin([5, 6, 7, 8])
    ]
    .sort_values("window_id")
    .copy()
)

# Validation
assert len(origin_signal_audit_df) == 17
assert len(focus_windows_df) == 4

assert origin_signal_audit_df["window_id"].is_unique

assert origin_signal_audit_df[
    [
        "origin_promotion_share_lag_1",
        "origin_promotion_share_3m_avg",
        "origin_promoted_items_3m_sum",
        "additional_absolute_error",
        "promotion_share_vs_3m_avg_pp",
    ]
].notna().all().all()

assert np.isclose(
    origin_signal_audit_df["additional_absolute_error"].sum(),
    overall_additional_error
)

print("Historical promotion signal investigation: PASSED")
print("Forecast origins investigated:", len(origin_signal_audit_df))
print("High-error focus windows:", len(focus_windows_df))

display(
    focus_windows_df.round({
        "origin_promotion_share_lag_1": 2,
        "origin_promotion_share_3m_avg": 2,
        "origin_promoted_items_3m_sum": 0,
        "additional_absolute_error": 2,
        "promotion_share_vs_3m_avg_pp": 2,
    })
)

Historical promotion signal investigation: PASSED
Forecast origins investigated: 17
High-error focus windows: 4


,window_id,train_end,origin_promotion_share_lag_1,origin_promotion_share_3m_avg,origin_promoted_items_3m_sum,additional_absolute_error,promotion_share_vs_3m_avg_pp
4,5,2016-04-01,29.41,24.43,580643,8139605.20,4.98
5,6,2016-05-01,39.81,31.21,767121,18566053.71,8.61
6,7,2016-06-01,40.86,36.69,870747,13106455.25,4.16
7,8,2016-07-01,40.91,40.53,920040,8121992.94,0.38


## Step 15 — Export Validated Forecast Evaluation Reports

Save rolling-origin Ridge predictions, four-model accuracy
metrics, and window-level error diagnostics.

Verify that the saved files contain the expected records
and that the reported accuracy metrics reconcile with the
in-memory calculations.

These reports document a historical evaluation experiment,
not a production-ready forecasting model.

In [16]:
# Create the report folder
report_dir = (
    PROJECT_ROOT
    / "reports"
    / "forecasting_feature_evaluation"
)

report_dir.mkdir(parents=True, exist_ok=True)

# Define output files
forecast_path = (
    report_dir / "ridge_rolling_origin_forecasts.csv"
)

comparison_path = (
    report_dir / "four_model_accuracy_comparison.csv"
)

diagnostics_path = (
    report_dir / "ridge_window_error_diagnostics.csv"
)

# Export reports
ridge_forecasts_df.to_csv(forecast_path, index=False)

four_model_comparison_df.to_csv(
    comparison_path, index=False
)

window_error_comparison_df.to_csv(
    diagnostics_path, index=False
)

# Read the saved files back for validation
saved_forecasts = pd.read_csv(forecast_path)
saved_comparison = pd.read_csv(comparison_path)
saved_diagnostics = pd.read_csv(diagnostics_path)

# Structural checks
assert len(saved_forecasts) == 102
assert len(saved_comparison) == 4
assert len(saved_diagnostics) == 17

assert saved_forecasts["model"].nunique() == 2
assert saved_comparison["model"].nunique() == 4
assert saved_diagnostics["window_id"].nunique() == 17

# Reconcile saved accuracy metrics
assert np.allclose(
    saved_comparison["WAPE_pct"].to_numpy(),
    four_model_comparison_df["WAPE_pct"].to_numpy()
)

assert np.allclose(
    saved_forecasts["predicted_sales"].to_numpy(),
    ridge_forecasts_df["predicted_sales"].to_numpy()
)

assert np.isclose(
    saved_diagnostics["additional_absolute_error"].sum(),
    overall_additional_error
)

print("Forecast evaluation report export: PASSED")
print("Saved forecast records:", len(saved_forecasts))
print("Saved model comparisons:", len(saved_comparison))
print("Saved window diagnostics:", len(saved_diagnostics))

print("\nFiles saved:")
print(forecast_path.relative_to(PROJECT_ROOT))
print(comparison_path.relative_to(PROJECT_ROOT))
print(diagnostics_path.relative_to(PROJECT_ROOT))

Forecast evaluation report export: PASSED
Saved forecast records: 102
Saved model comparisons: 4
Saved window diagnostics: 17

Files saved:
reports\forecasting_feature_evaluation\ridge_rolling_origin_forecasts.csv
reports\forecasting_feature_evaluation\four_model_accuracy_comparison.csv
reports\forecasting_feature_evaluation\ridge_window_error_diagnostics.csv


## Step 16 — Final Model Evaluation and Business Interpretation

### 1. Business Objective

Evaluate whether advance-known calendar features and
forecast-origin-safe historical promotion features improve
monthly retail sales forecasting accuracy.

The experiment uses 17 expanding rolling-origin windows,
a three-month forecast horizon, and 51 matched forecast cases
per model.

### 2. Model Performance

| Model | MAE | RMSE | WAPE | Bias |
|---|---:|---:|---:|---:|
| 50/50 Blended | 2,067,404.61 | 2,737,131.72 | 8.36% | -6.45% |
| ETS Additive | 2,098,242.04 | 2,542,024.93 | 8.48% | +0.96% |
| Calendar Only Ridge | 2,475,681.41 | 3,007,249.48 | 10.01% | -7.85% |
| Calendar + Promotion Ridge | 3,028,155.12 | 3,789,357.27 | 12.24% | -1.34% |

### 3. Key Findings

**Finding 1 — Best model by WAPE**

The 50/50 Blended benchmark achieved the lowest WAPE
at 8.36%.

ETS Additive achieved the lowest RMSE and the smallest
absolute bias among the four evaluated models.

**Finding 2 — Calendar features**

Calendar Only Ridge achieved 10.01% WAPE.

It did not outperform the existing blended or ETS
benchmarks.

**Finding 3 — Promotion features**

Adding historical promotion features increased Ridge
WAPE from 10.01% to 12.24%.

However, the promotion-enhanced model reduced the
magnitude of overall negative forecast bias.

**Finding 4 — Error distribution**

Promotion features improved 30 of 51 individual forecast
cases and 10 of 17 rolling-origin windows.

Nevertheless, larger errors in other windows produced
28,176,159.26 additional absolute-error units overall.

**Finding 5 — Error concentration**

The five largest deteriorating windows accounted for
94.77% of total positive error deterioration.

Four consecutive high-error origins occurred between
April and July 2016.

**Finding 6 — Historical promotion patterns**

During April–July 2016, recorded promotion share increased
from 29.41% to 40.91%.

These changing historical promotion signals coincided
with larger forecast errors, but this does not establish
causation.

### 4. Business Recommendation

Retain the 50/50 Blended model as the current preferred
candidate when minimizing WAPE is the primary objective.

Consider ETS Additive when reducing large forecast errors
or overall bias is more important.

Do not replace the existing benchmarks with either Ridge
model based on this experiment.

Further model development should investigate more robust
feature representations, model specifications, and
independent validation periods.

### 5. Limitations

1. The dataset contains 55 complete historical months
   from January 2013 through July 2017.

2. The 51 evaluation cases arise from overlapping
   rolling-origin forecasts and are not independent months.

3. Historical promotion features describe recorded
   promotion activity, not future promotion plans.

4. The earliest 15 months have uncertain promotion
   coverage and should not be interpreted as confirmed
   zero-promotion months.

5. Promotion-record share is not a discount percentage,
   and promoted-item counts are not promotion rates.

6. The Ridge models were evaluated with a fixed
   regularization parameter and were not independently
   optimized.

7. The analysis establishes predictive performance,
   not the causal effect of promotions on sales.

8. The preferred model is a historical evaluation
   candidate, not a production deployment approval.

### 6. Reproducibility

The following validated reports are available:

- `reports/forecasting_feature_evaluation/ridge_rolling_origin_forecasts.csv`
- `reports/forecasting_feature_evaluation/four_model_accuracy_comparison.csv`
- `reports/forecasting_feature_evaluation/ridge_window_error_diagnostics.csv`

All Ridge predictions were generated using historical
training information available at their forecast origins.